Restructuring the table. We need to add the tracking column and a surrogate key. Since SCD2 will hold multiple rows per employee over time.

In [0]:
%sql
--creating the SCD2-ready table from the current dim_employee (treating today as the start date
CREATE OR REPLACE TABLE ibm_hr.silver.dim_employee_scd2 AS
SELECT
  monotonically_increasing_id() AS employee_sk,
  employee_id,
  Age,
  Gender,
  MaritalStatus,
  Department,
  JobRole,
  JobLevel,
  Education,
  EducationField,
  BusinessTravel,
  OverTime,
  Attrition,
  CAST('2026-01-01' AS DATE) AS from_date,
  CAST(NULL AS DATE) AS to_date,
  TRUE AS in_use_flag
FROM ibm_hr.silver.dim_employee

In [0]:
#checking
spark.sql("SELECT * FROM ibm_hr.silver.dim_employee_scd2 LIMIT 5").show()

losing out the old version (set to_date and in_use_flag = false for rows whose data has changed)

In [0]:
%sql
MERGE INTO ibm_hr.silver.dim_employee_scd2 AS target
USING (SELECT 1 AS employee_id, 'Sales' AS Department, 'Sales Executive' AS JobRole) AS source
ON target.employee_id = source.employee_id
   AND target.in_use_flag = true
   AND (target.Department <> source.Department OR target.JobRole <> source.JobRole)
WHEN MATCHED THEN
  UPDATE SET
    target.to_date = CAST('2026-06-19' AS DATE),
    target.in_use_flag = false

Inserting the new version (the new row representing the change):

In [0]:
%sql
INSERT INTO ibm_hr.silver.dim_employee_scd2
SELECT
  (SELECT MAX(employee_sk) + 1 FROM ibm_hr.silver.dim_employee_scd2) AS employee_sk,
  employee_id, Age, Gender, MaritalStatus,
  'Sales' AS Department,
  'Sales Executive' AS JobRole,
  JobLevel, Education, EducationField, BusinessTravel, OverTime, Attrition,
  CAST('2026-06-19' AS DATE) AS from_date,
  CAST(NULL AS DATE) AS to_date,
  TRUE AS in_use_flag
FROM ibm_hr.silver.dim_employee_scd2
WHERE employee_id = 1 AND in_use_flag = false
ORDER BY to_date DESC
LIMIT 1

In [0]:
spark.sql("SELECT * FROM ibm_hr.silver.dim_employee_scd2 WHERE employee_id = 1 ORDER BY from_date").show()